In [ ]:
import json
import re
from datasets import load_from_disk
from collections import defaultdict

sft_path = "/content/drive/MyDrive/vlm-finetuning-project1/results/inference/unified-sft-v1_test/repair_applied/predictions_repaired.jsonl"
dataset_path = "/content/drive/MyDrive/vlm-finetuning-project1/datasets/processed"

def strip_fences(text):
    match = re.search(r"```(?:json)?(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    return match.group(1).strip() if match else text.strip()

print("Loading dataset...")
ds = load_from_disk(dataset_path)
test_ds = ds["test"]

Loading dataset...


In [ ]:

print("Loading predictions...")
predictions = {}
with open(sft_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            record = json.loads(line)
            img_id = str(record.get("image_id", ""))
            raw_str = record.get("raw_output", "")

            try:
                # Try parsing raw (works for the 35 fixed_valid ones without fences)
                parsed_pred = json.loads(raw_str)
                predictions[img_id] = parsed_pred
            except Exception:
                # Try stripping fences (works for the 2969 valid_raw ones WITH fences)
                try:
                    parsed_pred = json.loads(strip_fences(raw_str))
                    predictions[img_id] = parsed_pred
                except Exception:
                    pass # Truly unparseable

print(f"Loaded {len(predictions)} predictions.")

Loading predictions...
Loaded 3004 predictions.


In [ ]:
record

{'image_id': '0004164',
 'raw_output': '```json\n{"caption":"The image shows two large red rollers on the left side with one operator visible inside operating it. In the center, there is a yellow road roller being operated by another worker standing beside it.","rule_1_violation":null,"rule_2_violation":null,"rule_3_violation":null,"rule_4_violation":null,"excavator":[[570,360,918,710]],"rebar":[],"worker_with_white_hard_hat":[]}\n```',
 'sample': {'image_id': '0004164',
  'image_caption': 'The image shows a construction site with two workers visible on the left, standing in front of a gantry crane. To the right of the workers, there is a yellow drum roller and a yellow Hyundai excavator. The ground appears muddy with tracks from construction vehicles.',
  'illumination': 'normal lighting',
  'camera_distance': 'mid distance',
  'view': 'elevation view',
  'quality_of_info': 'rich info',
  'rule_1_violation': None,
  'rule_2_violation': None,
  'rule_3_violation': None,
  'rule_4_viola

In [ ]:
test_ds[0]

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=320x240>,
 'image_id': '0000051',
 'image_caption': 'The bucket of an excavator is shoveling rocks.',
 'illumination': 'normal lighting',
 'camera_distance': 'mid distance',
 'view': 'elevation view',
 'quality_of_info': 'poor info',
 'rule_1_violation': None,
 'rule_2_violation': None,
 'rule_3_violation': None,
 'rule_4_violation': None,
 'excavator': [[0.52, 0.0, 0.91, 0.82]],
 'rebar': [],
 'worker_with_white_hard_hat': [],
 'resolution': 76800}

In [ ]:
# Extracting Rule 1 False Negatives
# A False Negative (FN) means Ground Truth HAS Rule 1, but Model PREDICTED NO Rule 1.

rule_1_fn_cases = []
rule_1_tp_cases = [] # True Positives for comparison

for example in test_ds:
    img_id = str(example.get("image_id", example.get("id", "")))

    if img_id not in predictions:
        print(f"Warning: Missing prediction for image {img_id}")
        continue

    pred = predictions[img_id]

    # Check GT
    gt_r1 = example.get("rule_1_violation")
    has_gt_r1 = bool(gt_r1)

    # Check Pred
    pred_r1 = pred.get("rule_1_violation")
    has_pred_r1 = bool(pred_r1)

    if has_gt_r1:
        # Extract the ground truth reason string
        gt_reason = gt_r1.get("reason", "") if isinstance(gt_r1, dict) else str(gt_r1)

        if not has_pred_r1:
            rule_1_fn_cases.append({"image_id": img_id, "reason": gt_reason})
        else:
            rule_1_tp_cases.append({"image_id": img_id, "reason": gt_reason})

print(f"Total Rule 1 Ground Truth Cases: {len(rule_1_fn_cases) + len(rule_1_tp_cases)}")
print(f"Rule 1 True Positives (Hits): {len(rule_1_tp_cases)}")
print(f"Rule 1 False Negatives (Misses): {len(rule_1_fn_cases)}")
print(f"Rule 1 Recall: {len(rule_1_tp_cases) / (len(rule_1_fn_cases) + len(rule_1_tp_cases)):.1%}")

Total Rule 1 Ground Truth Cases: 323
Rule 1 True Positives (Hits): 128
Rule 1 False Negatives (Misses): 195
Rule 1 Recall: 39.6%


In [ ]:
# Categorize by PPE Sub-Type via Keywords
# We will define keyword clusters to automatically classify the GT reason strings.

categories = {
    "Hard Hat": ["hat", "helmet", "head"],
    "Footwear": ["shoe", "boot", "foot", "feet", "toe"],
    "Gloves": ["glove", "hand"],
    "High-Vis / Vest": ["vest", "high-vis", "visibility", "reflective", "jacket", "night", "dark"],
    "Face / Eye Protection": ["face", "mask", "goggle", "shield", "glass", "weld", "grind", "drill", "cut", "eye"],
    "Clothing (General)": ["shirt", "pant", "sleeve", "cloth", "short", "bare", "topless"],
}

def categorize_reason(reason_str):
    reason_lower = reason_str.lower()
    matched_cats = []

    for cat_name, keywords in categories.items():
        if any(kw in reason_lower for kw in keywords):
            matched_cats.append(cat_name)

    if not matched_cats:
        return ["Other / Unspecified"]
    return matched_cats

# Analyze False Negatives (Where the model failed)
fn_category_counts = defaultdict(int)
fn_examples_by_cat = defaultdict(list)

for case in rule_1_fn_cases:
    cats = categorize_reason(case["reason"])
    for c in cats:
        fn_category_counts[c] += 1
        if len(fn_examples_by_cat[c]) < 3: # Save up to 3 examples for display
            fn_examples_by_cat[c].append(case)

# Analyze True Positives (Where the model succeeded, for comparison)
tp_category_counts = defaultdict(int)
for case in rule_1_tp_cases:
    cats = categorize_reason(case["reason"])
    for c in cats:
        tp_category_counts[c] += 1

In [ ]:
# Print Statistical Report & Examples

print("=== RULE 1 FALSE NEGATIVE ANALYSIS (Where the model failed) ===")
total_fns = len(rule_1_fn_cases)

# Sort categories by highest failure count
sorted_fn_cats = sorted(fn_category_counts.items(), key=lambda x: x[1], reverse=True)

for cat, count in sorted_fn_cats:
    pct = (count / total_fns) * 100
    print(f"\n[{cat}]: {count} Misses ({pct:.1f}% of total Rule 1 misses)")

    # Print examples
    print("  Examples of GT Reasons the model missed:")
    for ex in fn_examples_by_cat[cat]:
        print(f"    - (Img {ex['image_id']}): {ex['reason']}")

print("\n" + "="*50 + "\n")
print("=== COMPARISON: MISS RATE BY PPE SUB-TYPE ===")
# Compute recall per sub-category (How many Hard Hat cases did we hit vs miss?)
all_categories = set(fn_category_counts.keys()).union(set(tp_category_counts.keys()))

for cat in sorted(all_categories):
    tp = tp_category_counts.get(cat, 0)
    fn = fn_category_counts.get(cat, 0)
    total = tp + fn

    recall = (tp / total) * 100 if total > 0 else 0
    miss_rate = (fn / total) * 100 if total > 0 else 0

    print(f"{cat:25} | Total GT: {total:<4} | Hits: {tp:<4} | Misses: {fn:<4} | Miss Rate: {miss_rate:>5.1f}%")

=== RULE 1 FALSE NEGATIVE ANALYSIS (Where the model failed) ===

[Hard Hat]: 155 Misses (79.5% of total Rule 1 misses)
  Examples of GT Reasons the model missed:
    - (Img 0001444): The two workers sitting on top of a pile of rocks on the left are not wearing hard hats.
    - (Img 0004680): The two workers on top of the rock pile are not wearing hard hats.
    - (Img 0000571): The worker is not wearing a hard hat.

[Clothing (General)]: 31 Misses (15.9% of total Rule 1 misses)
  Examples of GT Reasons the model missed:
    - (Img 0003546): The person on the left is wearing shorts.
    - (Img 0001531): The worker on the left is not wearing a hard hat but a straw hat. The pants of the worker on the right do not cover his ankles.
    - (Img 0000599): A worker in the lower left is wearing short pants.

[High-Vis / Vest]: 23 Misses (11.8% of total Rule 1 misses)
  Examples of GT Reasons the model missed:
    - (Img 0004711): The two workers are not wearing high-visibility vests when workin

In [ ]:
import json
from datasets import load_from_disk
from collections import defaultdict

sft_path = "/content/drive/MyDrive/vlm-finetuning-project1/results/inference/unified-sft-v1_test/repair_applied/predictions_repaired.jsonl"
dataset_path = "/content/drive/MyDrive/vlm-finetuning-project1/datasets/processed"

print("Loading dataset...")
ds = load_from_disk(dataset_path)
test_ds = ds["test"]

import re

def strip_fences(text):
    match = re.search(r"```(?:json)?(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    return match.group(1).strip() if match else text.strip()

print("Loading predictions...")
predictions = {}
with open(sft_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            record = json.loads(line)
            img_id = str(record.get("image_id", ""))
            raw_str = record.get("raw_output", "")

            try:
                parsed_pred = json.loads(raw_str)
                predictions[img_id] = parsed_pred
            except Exception:
                try:
                    parsed_pred = json.loads(strip_fences(raw_str))
                    predictions[img_id] = parsed_pred
                except Exception:
                    pass # Skip unparseable

print(f"Loaded {len(predictions)} predictions.")

Loading dataset...
Loading predictions...
Loaded 3004 predictions.


In [ ]:
# Extract Rule 1 Cases with Metadata
# We only care about images that ACTUALLY have a Rule 1 violation in Ground Truth

rule_1_cases = []

for example in test_ds:
    img_id = str(example.get("image_id", example.get("id", "")))

    if img_id not in predictions:
        print(f"Warning: Missing prediction for image {img_id}")
        continue

    pred = predictions[img_id]

    # Check GT
    has_gt_r1 = bool(example.get("rule_1_violation"))

    # Check Pred
    has_pred_r1 = bool(pred.get("rule_1_violation"))

    if has_gt_r1:
        # Determine if it was a Hit (True Positive) or Miss (False Negative)
        status = "Hit" if has_pred_r1 else "Miss"

        # Extract the 4 metadata fields
        rule_1_cases.append({
            "image_id": img_id,
            "status": status,
            "illumination": example.get("illumination", "unknown"),
            "camera_distance": example.get("camera_distance", "unknown"),
            "view": example.get("view", "unknown"),
            "quality_of_info": example.get("quality_of_info", "unknown")
        })

print(f"Total Rule 1 Ground Truth Cases Found: {len(rule_1_cases)}")

Total Rule 1 Ground Truth Cases Found: 323


In [ ]:
# Stratified Analysis Logic

metadata_fields = ["illumination", "camera_distance", "view", "quality_of_info"]
stratified_stats = {field: defaultdict(lambda: {"hits": 0, "misses": 0, "total": 0}) for field in metadata_fields}

for case in rule_1_cases:
    for field in metadata_fields:
        val = str(case[field]).strip()
        if not val:
            val = "unknown"

        stratified_stats[field][val]["total"] += 1
        if case["status"] == "Hit":
            stratified_stats[field][val]["hits"] += 1
        else:
            stratified_stats[field][val]["misses"] += 1

In [ ]:
print("=== RULE 1 PERCEPTUAL DIFFICULTY (STRATIFIED ANALYSIS) ===\n")

for field in metadata_fields:
    print(f"--- Breakdown by: {field.upper()} ---")

    # Sort buckets by total cases (descending)
    sorted_buckets = sorted(stratified_stats[field].items(), key=lambda x: x[1]["total"], reverse=True)

    for bucket_name, stats in sorted_buckets:
        total = stats["total"]
        hits = stats["hits"]
        misses = stats["misses"]

        miss_rate = (misses / total) * 100 if total > 0 else 0
        hit_rate = (hits / total) * 100 if total > 0 else 0

        print(f"  {bucket_name:20} | Total GT: {total:<4} | Hits: {hits:<4} | Misses: {misses:<4} | Miss Rate: {miss_rate:>5.1f}%")

    print("-" * 65 + "\n")

=== RULE 1 PERCEPTUAL DIFFICULTY (STRATIFIED ANALYSIS) ===

--- Breakdown by: ILLUMINATION ---
  normal lighting      | Total GT: 247  | Hits: 102  | Misses: 145  | Miss Rate:  58.7%
  underexposed         | Total GT: 42   | Hits: 19   | Misses: 23   | Miss Rate:  54.8%
  night                | Total GT: 20   | Hits: 3    | Misses: 17   | Miss Rate:  85.0%
  overexposed          | Total GT: 14   | Hits: 4    | Misses: 10   | Miss Rate:  71.4%
-----------------------------------------------------------------

--- Breakdown by: CAMERA_DISTANCE ---
  short distance       | Total GT: 156  | Hits: 73   | Misses: 83   | Miss Rate:  53.2%
  mid distance         | Total GT: 144  | Hits: 48   | Misses: 96   | Miss Rate:  66.7%
  long distance        | Total GT: 23   | Hits: 7    | Misses: 16   | Miss Rate:  69.6%
-----------------------------------------------------------------

--- Breakdown by: VIEW ---
  elevation view       | Total GT: 314  | Hits: 126  | Misses: 188  | Miss Rate:  59.9%
  